# Preprocessing

Before matching, raw choices need to be validated: project codes must exist, projects must be open to the student's course, and duplicate picks of the same project need to be collapsed to the best rank. This notebook applies `matched.preprocess` to the data introduced in [The data](01-data.ipynb).

We start by reproducing the loading and reshaping steps from the previous notebook:

In [ ]:
import pathlib

import pandas as pd

import matched

raw = pathlib.Path("./raw")

projects = pd.read_csv(raw / "projects.csv").set_index("code")
choices_wide = pd.read_csv(raw / "internal-choices.csv")

choice_cols = [c for c in choices_wide.columns if c.startswith("choice")]
choices = choices_wide.melt(
    id_vars=["username", "course", "score"],
    value_vars=choice_cols,
    var_name="choice",
    value_name="code",
)
choices["choice"] = choices["choice"].str.removeprefix("choice").astype(int)
choices = choices.sort_values(["username", "choice"]).reset_index(drop=True)

len(choices)

### Invalid project codes

`filter_invalid_code` drops choices for a project `code` that isn't in the known project list - for example a typo, or a project withdrawn after the form was opened. We check what changes:

In [ ]:
before = choices.copy()
choices = choices.pipe(matched.filter_invalid_code, valid_codes=projects.index)

before[~before.index.isin(choices.index)]

Nothing is dropped here - every code in this dataset exists in `projects.csv`. In practice, this step guards against typos or a project withdrawn after the form was opened.

### Projects not offered to the student's course

`filter_invalid_course` drops a choice if the project isn't offered to the student's `course` (checked against the boolean `course1`/`course2` columns in `projects.csv`).

In [ ]:
before = choices.copy()
choices = choices.pipe(matched.filter_invalid_course, projects=projects)

before[~before.index.isin(choices.index)]

Three choices are removed. `kl112`, for example, is on `course2` but had picked `alfa-012` - a project only open to `course1` - as their first choice; after preprocessing, their first *valid* choice becomes whatever they ranked next. This matters for the outcome: in [The algorithm](03-algorithm.ipynb) we'll see `kl112` allocated their second choice rather than the (invalid) first one.

### Duplicate picks

`deduplicate` collapses repeated `(username, code)` picks, keeping only the best (lowest) `choice` rank - e.g. if a student accidentally listed the same project twice.

In [ ]:
before = choices.copy()
choices = choices.pipe(matched.deduplicate)

len(before), len(choices)

No duplicates in this dataset, so nothing changes here - but the same pipeline handles it automatically when they do occur.

### Putting it together

The three steps compose cleanly with `.pipe`, in this order - code validity, then course validity, then de-duplication:

In [ ]:
clean_choices = (
    choices_wide.melt(
        id_vars=["username", "course", "score"],
        value_vars=choice_cols,
        var_name="choice",
        value_name="code",
    )
    .assign(choice=lambda df: df.choice.str.removeprefix("choice").astype(int))
    .pipe(matched.filter_invalid_code, valid_codes=projects.index)
    .pipe(matched.filter_invalid_course, projects=projects)
    .pipe(matched.deduplicate)
)

clean_choices.sort_values(["username", "choice"]).head(8)

`clean_choices` is now ready to allocate. [The algorithm](03-algorithm.ipynb) walks through how `match` turns this into an allocation, and what happens when a project is oversubscribed.